In [1]:
from typing import Any, Dict
import json

GUIDELINE_KEYS = [
    "admissibility_guide",
    "application_notes",
    "common_mistakes",
    "case_law",
]


def _render_prompt_content(node: Any, indent: int = 0) -> str:
    pad = "  " * indent
    output: list[str] = []

    # Leaf field: has user value + prompts/guidelines
    if isinstance(node, dict) and "value" in node:
        output.append(f"{pad}given_value: {json.dumps(node.get('value'), ensure_ascii=False)}")

        prompts = node.get("prompts", {})
        if isinstance(prompts, dict):
            output.append(f"{pad}verification_guidelines:")
            for key in GUIDELINE_KEYS:
                output.append(
                    f"{pad}  {key}: {json.dumps(prompts.get(key), ensure_ascii=False)}"
                )

        return "\n".join(output)

    # Nested section / subsection
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "prompts":
                continue
            output.append(f"{pad}{key}:")
            output.append(_render_prompt_content(value, indent + 1))
        return "\n".join(output)

    return f"{pad}given_value: {json.dumps(node, ensure_ascii=False)}"


def _build_single_section_prompt(section_name: str, section_body: Any) -> str:
    section_structure = _extract_output_structure(section_body)

    return f"""
You are completing the European Court of Human Rights application form.

Section: {section_name}
{"-" * 40}

Task:
For each field below:
1. Read the user's given value.
2. Check it against the verification guidelines.
3. Correct the value if it is incomplete, noisy, wrongly formatted, or obviously incorrect.
4. If the value is missing and cannot be inferred, use null.
5. Explain briefly what was wrong, if anything.

Input values and verification guidelines:
{_render_prompt_content(section_body, indent=0)}

Return ONLY valid JSON.

The JSON must keep the same section structure as below.

For every field, return this object format:
{{
  "given_value": "...",
  "corrected_value": "...",
  "problem_in_given_value": "...",
  "verification_summary": "...",
  "confidence": "high | medium | low"
}}

Required output structure:
{json.dumps(section_structure, ensure_ascii=False, indent=2)}
""".strip()


def _extract_output_structure(node: Any) -> Any:
    """
    Keeps the same nesting as the form, but replaces every field
    with the required LLM output object.
    """

    if isinstance(node, dict) and "value" in node:
        return {
            "given_value": node.get("value"),
            "corrected_value": None,
            "problem_in_given_value": None,
            "verification_summary": None,
            "confidence": None,
        }

    if isinstance(node, dict):
        return {
            key: _extract_output_structure(value)
            for key, value in node.items()
            if key != "prompts"
        }

    return {
        "given_value": node,
        "corrected_value": None,
        "problem_in_given_value": None,
        "verification_summary": None,
        "confidence": None,
    }


def build_section_prompts(prompted_output: Dict[str, Any]) -> Dict[str, str]:
    section_prompts: Dict[str, str] = {}

    for section_name, section_body in prompted_output.items():
        section_prompts[section_name] = _build_single_section_prompt(
            section_name=section_name,
            section_body=section_body,
        )

    return section_prompts

In [ ]:
from typing import Any, Dict
import json

GUIDELINE_KEYS = [
    "admissibility_guide",
    "application_notes",
    "common_mistakes",
    "case_law",
]


def _render_prompt_content(node: Any, indent: int = 0) -> str:
    pad = "  " * indent
    output: list[str] = []

    # Leaf field: has user value + prompts/guidelines
    if isinstance(node, dict) and "value" in node:
        output.append(f"{pad}given_value: {json.dumps(node.get('value'), ensure_ascii=False)}")

        prompts = node.get("prompts", {})
        if isinstance(prompts, dict):
            output.append(f"{pad}verification_guidelines:")
            for key in GUIDELINE_KEYS:
                output.append(
                    f"{pad}  {key}: {json.dumps(prompts.get(key), ensure_ascii=False)}"
                )

        return "\n".join(output)

    # Nested section / subsection
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "prompts":
                continue
            output.append(f"{pad}{key}:")
            output.append(_render_prompt_content(value, indent + 1))
        return "\n".join(output)

    return f"{pad}given_value: {json.dumps(node, ensure_ascii=False)}"


def _build_single_section_prompt(section_name: str, section_body: Any) -> str:
    section_structure = _extract_output_structure(section_body)

    return f"""
You are completing the European Court of Human Rights application form.

Section: {section_name}
{"-" * 40}

Task:
For each field below:
1. Read the user's given value.
2. Check it against the verification guidelines.
3. Correct the value if it is incomplete, noisy, wrongly formatted, or obviously incorrect.
4. If the value is missing and cannot be inferred, use null.
5. Explain briefly what was wrong, if anything.

Input values and verification guidelines:
{_render_prompt_content(section_body, indent=0)}

Return ONLY valid JSON.

The JSON must keep the same section structure as below.

For every field, return this object format:
{{
  "given_value": "...",
  "corrected_value": "...",
  "problem_in_given_value": "...",
  "verification_summary": "...",
  "confidence": "high | medium | low"
}}

Required output structure:
{json.dumps(section_structure, ensure_ascii=False, indent=2)}
""".strip()


def _extract_output_structure(node: Any) -> Any:
    """
    Keeps the same nesting as the form, but replaces every field
    with the required LLM output object.
    """

    if isinstance(node, dict) and "value" in node:
        return {
            "given_value": node.get("value"),
            "corrected_value": None,
            "problem_in_given_value": None,
            "verification_summary": None,
            "confidence": None,
        }

    if isinstance(node, dict):
        return {
            key: _extract_output_structure(value)
            for key, value in node.items()
            if key != "prompts"
        }

    return {
        "given_value": node,
        "corrected_value": None,
        "problem_in_given_value": None,
        "verification_summary": None,
        "confidence": None,
    }


def build_section_prompts(prompted_output: Dict[str, Any]) -> Dict[str, str]:
    section_prompts: Dict[str, str] = {}

    for section_name, section_body in prompted_output.items():
        section_prompts[section_name] = _build_single_section_prompt(
            section_name=section_name,
            section_body=section_body,
        )

    return section_prompts

In [2]:
with open("app data/form_data_with_prompts.json", "r", encoding="utf-8") as f:
    form_data_with_prompts_dict = json.load(f)

section_prompts = build_section_prompts(form_data_with_prompts_dict)

In [9]:
import json
from pathlib import Path


folder = Path("app data")
folder.mkdir(exist_ok=True)

file_path = folder / "final_prompts_per_section.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(section_prompts, f, indent=2)


In [8]:
import tiktoken
import pandas as pd


def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """
    Count tokens using OpenAI tokenizer.
    """

    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")

    return len(encoding.encode(text))


def calculate_prompt_tokens(
    section_prompts: dict[str, str],
    model: str = "gpt-4o"
) -> pd.DataFrame:
    """
    Calculate token counts for each section prompt.
    """

    rows = []

    for section_name, prompt in section_prompts.items():
        token_count = count_tokens(prompt, model=model)

        rows.append({
            "section": section_name,
            "tokens": token_count,
            "characters": len(prompt),
        })

    df = pd.DataFrame(rows)

    df["tokens"] = df["tokens"].astype(int)
    df["characters"] = df["characters"].astype(int)

    df = df.sort_values("tokens", ascending=False)

    total_tokens = df["tokens"].sum()

    print(f"\nTotal tokens across all prompts: {total_tokens:,}\n")

    return df


# ---------------------------------------
# Example usage
# ---------------------------------------

token_df = calculate_prompt_tokens(
    section_prompts,
    model="gpt-4o"
)
token_df


Total tokens across all prompts: 21,533



,section,tokens,characters
0,A. Applicant,4328,21032
2,C. Representative (Individual applicant),3852,18684
3,D. Representative (Organisation),3287,16139
4,E. Statement of facts,2900,13265
5,F. Alleged violations,1691,7505
6,G. Admissibility,1493,6691
9,J. Final declaration,1380,6567
7,H. Other international proceedings,1336,6251
1,B. States,707,2624
8,I. List of accompanying documents,559,2634
